# Day 031 Project Solution — Resilient Batch Processor

A hardened batch processor with retry, dead-letter queue, and AI incident reporting.

In [ ]:
import ollama
import time
from datetime import datetime


import time


def retry(fn, max_attempts: int = 3, base_delay: float = 1.0, backoff: float = 2.0):
    last_error = None
    for attempt in range(max_attempts):
        try:
            return fn()
        except Exception as e:
            last_error = e
            if attempt < max_attempts - 1:
                time.sleep(base_delay * (backoff ** attempt))
    raise last_error


from datetime import datetime


class DeadLetterQueue:
    def __init__(self):
        self._items: list = []

    def add(self, item, error: str, context: dict | None = None) -> None:
        self._items.append({
            "item":     item,
            "error":    error,
            "context":  context or {},
            "added_at": datetime.now().isoformat(),
        })

    def drain(self) -> list:
        items, self._items = self._items, []
        return items

    def peek(self) -> list:
        return list(self._items)

    def size(self) -> int:
        return len(self._items)


def resilient_step(
    name: str,
    fn,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
) -> dict:
    start      = time.time()
    last_error = None
    for attempt in range(max_attempts):
        try:
            result = fn()
            return {
                "name":       name,
                "status":     "ok",
                "result":     result,
                "error":      None,
                "duration_s": round(time.time() - start, 3),
                "attempts":   attempt + 1,
            }
        except Exception as e:
            last_error = e
            if attempt < max_attempts - 1:
                time.sleep(base_delay * (backoff ** attempt))
    return {
        "name":       name,
        "status":     "error",
        "result":     None,
        "error":      str(last_error),
        "duration_s": round(time.time() - start, 3),
        "attempts":   max_attempts,
    }


def process_batch_with_dlq(
    items: list,
    process_fn,
    dlq: DeadLetterQueue,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
) -> list:
    results = []
    for item in items:
        r = resilient_step(
            str(item),
            lambda i=item: process_fn(i),
            max_attempts=max_attempts,
            base_delay=base_delay,
            backoff=backoff,
        )
        if r["status"] == "error":
            dlq.add(item, r["error"])
        results.append(r)
    return results


def ai_resilience_report(
    batch_results: list,
    dlq_items: list,
    model: str = "llama3.2",
) -> str:
    total        = len(batch_results)
    passed       = sum(1 for r in batch_results if r["status"] == "ok")
    avg_attempts = (
        round(sum(r.get("attempts", 1) for r in batch_results) / total, 2)
        if total else 0.0
    )
    lines = [
        f"Batch run: {passed}/{total} items succeeded, "
        f"{len(dlq_items)} failed to DLQ.",
        f"Average attempts per item: {avg_attempts}.",
    ]
    for entry in dlq_items[:3]:
        lines.append(f"  DLQ: {entry['item']} \u2014 {entry['error']}")

    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a reliability engineer. "
                    "Summarise a batch processing run in 2\u20133 sentences. "
                    "Focus on the failure rate and recommend one concrete action."
                ),
            },
            {
                "role": "user",
                "content": "\n".join(lines) + "\n\nSummarise and recommend:",
            },
        ],
    )
    return response["message"]["content"]

## Action 1 — Process Batch with Simulated Transient and Permanent Failures

In [ ]:
_attempt_tracker = {}

def process_item(item):
    _attempt_tracker[item] = _attempt_tracker.get(item, 0) + 1
    if str(item).startswith('fail'):
        raise ValueError(f'invalid item: {item}')
    if str(item).startswith('retry') and _attempt_tracker[item] < 2:
        raise ConnectionError(f'transient on attempt {_attempt_tracker[item]}')
    return f'ok:{item}'

items = ['item_1', 'retry_2', 'item_3', 'fail_4', 'retry_5', 'item_6']
dlq   = DeadLetterQueue()

result = process_batch_with_dlq(
    items, process_item, dlq, max_attempts=3, base_delay=0.0
)

print('Batch results:')
for r in result:
    print(f"  {r['status']:7} {r['name']:12} attempts={r['attempts']}")

print(f'\nSucceeded: {sum(1 for r in result if r["status"]=="ok")}/{len(result)}')
print(f'DLQ size:  {dlq.size()}')

## Action 2 — Inspect DLQ

In [ ]:
dlq_items = dlq.drain()
print(f'DLQ items ({len(dlq_items)}):')
for entry in dlq_items:
    print(f"  item={entry['item']!r:15} error={entry['error']!r}")

assert dlq.size() == 0, 'DLQ should be empty after drain'
print('DLQ drained successfully')

## Action 3 — AI Incident Report

In [ ]:
report = ai_resilience_report(result, dlq_items)
print('Incident Report:')
print(report)

assert isinstance(report, str) and len(report) > 10
assert any(r['attempts'] >= 1 for r in result)

print('\nResilience complete!')